# 10.9.大规模预训练 Transformer（Large-Scale Pretraining with Transformers）

到目前为止，在我们进行的图像分类和机器翻译实验中，模型都是在包含输入-输出样例的数据集上*从零开始*（from scratch）训练的，以执行特定任务。例如，我们用英语-法语对（[10.7节](10.07_transformer.ipynb)）训练了一个Transformer，使得该模型能够将输入的英文文本翻译成法语。因此，每个模型都成为了一个*特定专家*（specific expert），即使数据分布发生微小变化也会受到影响（[4.9节](../04_pypto_multilayer_perceptrons/04.09_environment.ipynb) 环境和分布偏移）。为了获得泛化能力更好的模型，甚至是能力更强的*通才*（generalist）——可以在有适配或无适配的情况下执行多种任务——在大规模数据上*预训练*（pretraining）模型已变得越来越普遍。

给定更大的预训练数据，Transformer架构在模型规模和训练算力增加时表现更好，展现出卓越的*规模扩展*（scaling）行为。具体来说，基于Transformer的语言模型的性能与模型参数量、训练词元数和训练算力呈幂律关系 ([Kaplan et al., 2020](https://zh.d2l.ai/chapter_references/zreferences.html#kaplan2020scaling))。Transformer的可扩展性还体现在更大规模的视觉Transformer在更大数据上训练后性能的显著提升（在[10.8节](10.08_vision_transformer.ipynb)中讨论）。更近期的成功案例包括Gato，一个*通才*模型，可以玩Atari游戏、为图像生成描述、聊天并充当机器人 ([Reed et al., 2022](https://zh.d2l.ai/chapter_references/zreferences.html#reed2022generalist))。Gato是一个单一的Transformer，当在包括文本、图像、关节力矩和按钮按压等多种模态上预训练时，它扩展得很好。值得注意的是，所有这些多模态数据都被序列化为扁平的词元序列，Transformer可以像处理文本词元（[10.7节](10.07_transformer.ipynb)）或图像块（[10.8节](10.08_vision_transformer.ipynb)）一样处理它们。

在多模态数据上预训练Transformer取得巨大成功之前，Transformer已经在大量文本上进行了广泛的预训练。Transformer架构最初是为机器翻译提出的，其结构如图10.7.1所示，它由一个用于表示输入序列的编码器和一个用于生成目标序列的解码器组成。总体而言，Transformer可以以三种不同的模式使用：*仅编码器*、*编码器-解码器*和*仅解码器*。在本章的最后，我们将回顾这三种模式，并解释预训练Transformer中的规模扩展。

<div align="center">
 <img src="./images/transformer.svg" alt="图10.7.1 Transformer 架构" width="400">
 <br><small>图10.7.1 Transformer 架构</small>
</div>



## 10.9.1.仅编码器

当仅使用Transformer编码器时，输入词元序列被转换为相同数量的表示，这些表示可以进一步投影到输出（例如，分类）。Transformer编码器由自注意力层组成，其中所有输入词元相互关注。例如，图10.8.1中描绘的视觉Transformer是仅编码器的，它将输入图像块序列转换为特殊的“&lt;cls&gt;”词元的表示。

<div align="center">
 <img src="./images/vit.svg" alt="图10.8.1 视觉 Transformer 架构（英文版 Fig. 11.8.1）" width="400">
 <br><small>图10.8.1 视觉 Transformer 架构（英文版 Fig. 11.8.1）</small>
</div>

由于该表示依赖于所有输入词元，因此可以进一步投影为分类标签。这一设计受到了早期在文本上预训练的仅编码器Transformer——BERT（来自Transformer的双向编码器表示，Bidirectional Encoder Representations from Transformers）的启发 ([Devlin et al., 2018](https://zh.d2l.ai/chapter_references/zreferences.html#Devlin.Chang.Lee.ea.2018))。



### 10.9.1.1.预训练BERT

BERT使用*掩蔽语言建模*（masked language modeling）在文本序列上进行预训练：将带有随机掩蔽词元的输入文本馈入Transformer编码器，以预测被掩蔽的词元。

<div align="center">
 <img src="./images/bert-encoder-only.svg" alt="图10.9.1 BERT 掩蔽语言建模预训练（英文版 Fig. 11.9.1）" width="400">
 <br><small>图10.9.1 BERT 掩蔽语言建模预训练（英文版 Fig. 11.9.1）</small>
</div>

如图10.9.1所示，原始文本序列"I"、"love"、"this"、"red"、"car"前面添加了“&lt;cls&gt;”词元，且“&lt;mask&gt;”词元随机替换了"love"；然后，在预训练期间最小化掩蔽词元"love"与其预测之间的交叉熵损失。请注意，Transformer编码器的注意力模式没有任何约束（图10.9.1的右侧），因此所有词元都可以相互关注。由此，"love"的预测依赖于序列中其前后的输入词元。这就是BERT被称为"双向编码器"的原因。无需人工标注，来自书籍和维基百科的大规模文本数据即可用于预训练BERT。



### 10.9.1.2.微调BERT

预训练的BERT可以被*微调*（fine-tuned）到涉及单个文本或文本对的下游编码任务。在微调期间，可以向BERT添加具有随机参数的额外层：这些参数和预训练的BERT参数将被*更新*，以拟合下游任务的训练数据。

<div align="center">
 <img src="./images/bert-finetune-classification.svg" alt="图10.9.2 BERT 微调用于分类（英文版 Fig. 11.9.2）" width="400">
 <br><small>图10.9.2 BERT 微调用于分类（英文版 Fig. 11.9.2）</small>
</div>

展示了BERT微调用于情感分析。Transformer编码器是预训练的BERT，它以文本序列为输入，并将“&lt;cls&gt;”表示（输入的全局表示）馈入一个额外的全连接层以预测情感。在微调期间，通过基于梯度的算法最小化情感分析数据上预测与标签之间的交叉熵损失，其中额外层从零开始训练，同时更新预训练的BERT参数。
BERT不仅仅用于情感分析。拥有3.5亿参数的BERT从2500亿训练词元中学到的通用语言表示，推动了单文本分类、文本对分类或回归、文本标注和问答等自然语言任务的发展。

您可能注意到这些下游任务包括文本对理解。BERT预训练还有另一个损失，用于预测一个句子是否紧跟另一个句子。然而，当在2万亿词元上预训练同样规模的BERT变体RoBERTa时，这一损失后来被发现用处不大 ([Liu et al., 2019](https://zh.d2l.ai/chapter_references/zreferences.html#Liu.Ott.Goyal.ea.2019))。BERT的其他衍生模型改进了模型架构或预训练目标，例如ALBERT（强制参数共享） ([Lan et al., 2019](https://zh.d2l.ai/chapter_references/zreferences.html#lan2019albert))、SpanBERT（表示和预测文本跨度） ([Joshi et al., 2020](https://zh.d2l.ai/chapter_references/zreferences.html#joshi2020spanbert))、DistilBERT（通过知识蒸馏实现轻量化） ([Sanh et al., 2019](https://zh.d2l.ai/chapter_references/zreferences.html#sanh2019distilbert))以及ELECTRA（替换词元检测） ([Clark et al., 2019](https://zh.d2l.ai/chapter_references/zreferences.html#clark2019electra))。此外，BERT还启发了计算机视觉中的Transformer预训练，例如视觉Transformer ([Dosovitskiy et al., 2021](https://zh.d2l.ai/chapter_references/zreferences.html#Dosovitskiy.Beyer.Kolesnikov.ea.2021))、Swin Transformer ([Liu et al., 2021](https://zh.d2l.ai/chapter_references/zreferences.html#liu2021swin))和MAE（掩蔽自编码器） ([He et al., 2022](https://zh.d2l.ai/chapter_references/zreferences.html#he2022masked))。



## 10.9.2.编码器-解码器

由于Transformer编码器将输入词元序列转换为相同数量的输出表示，因此仅编码器模式无法像机器翻译那样生成任意长度的序列。正如最初为机器翻译提出的那样，Transformer架构可以配备一个解码器，该解码器以自回归方式逐词元地预测任意长度的目标序列，同时以编码器输出和解码器输出为条件：（i）为了以编码器输出为条件，编码器-解码器交叉注意力（图10.7.1中解码器的多头注意力）允许目标词元关注*所有*输入词元；（ii）以解码器输出为条件则通过所谓的*因果*注意力模式（掩蔽多头注意力，见图10.7.1中解码器的掩蔽多头注意力）实现（这个名称在文献中很常见，但具有误导性，因为它与对因果关系的严格研究关系不大），其中任何目标词元只能关注目标序列中的*过去*和*当前*词元。

<div align="center">
 <img src="./images/transformer.svg" alt="图10.7.1 Transformer 架构" width="400">
 <br><small>图10.7.1 Transformer 架构</small>
</div>

为了超越人工标注的机器翻译数据来预训练编码器-解码器Transformer，BART ([Lewis et al., 2019](https://zh.d2l.ai/chapter_references/zreferences.html#lewis2019bart))和T5 ([Raffel et al., 2020](https://zh.d2l.ai/chapter_references/zreferences.html#raffel2020exploring))是两个同时提出的编码器-解码器Transformer，在大规模文本语料库上预训练。两者都试图在其预训练目标中重建原始文本，前者强调对输入加噪（例如掩蔽、删除、置换和旋转），后者则强调多任务统一，并进行了全面的消融研究。



### 10.9.2.1.预训练T5

作为预训练Transformer编码器-解码器的一个例子，T5（文本到文本迁移Transformer，Text-to-Text Transfer Transformer）将许多任务统一为相同的文本到文本问题：对于任何任务，编码器的输入是任务描述（例如"Summarize"、":"）后跟任务输入（例如来自文章的词元序列），解码器预测任务输出（例如总结输入文章的词元序列）。为了以文本到文本的方式执行，T5被训练为在给定输入文本的条件下生成一些目标文本。

为了从任意原始文本中获得输入和输出，T5被预训练来预测连续跨度。具体来说，文本中的词元被随机替换为特殊词元，其中每个连续跨度被替换为相同的特殊词元。

<div align="center">
 <img src="./images/t5-encoder-decoder.svg" alt="图10.9.3 T5 跨度损坏预训练（英文版 Fig. 11.9.3）" width="400">
 <br><small>图10.9.3 T5 跨度损坏预训练（英文版 Fig. 11.9.3）</small>
</div>

图10.9.3给出了一个示例，其中原始文本是"I"、"love"、"this"、"red"、"car"。词元"love"、"red"、"car"被随机替换为特殊词元。由于"red"和"car"是一个连续跨度，它们被替换为相同的特殊词元。因此，输入序列是"I"、"&lt;X&gt;"、"this"、"&lt;Y&gt;"，目标序列是"&lt;X&gt;"、"love"、"&lt;Y&gt;"、"red"、"car"、"&lt;Z&gt;"，其中"&lt;Z&gt;"是另一个标记结束的特殊词元。如图10.9.3所示，解码器具有因果注意力模式，以防止其在序列预测期间关注未来词元。

在T5中，预测连续跨度也被称为重建损坏的文本。凭借这一目标，T5使用来自C4（巨型干净爬取语料库，Colossal Clean Crawled Corpus）数据的1万亿词元进行预训练，该数据包含来自网络的干净英文文本 ([Raffel et al., 2020](https://zh.d2l.ai/chapter_references/zreferences.html#raffel2020exploring))。



### 10.9.2.2.微调T5

与BERT类似，T5需要在特定任务的训练数据上进行微调（更新T5参数）才能执行该任务。与BERT微调的主要区别包括：（i）T5输入包含任务描述；（ii）T5可以凭借其Transformer解码器生成任意长度的序列；（iii）不需要额外的层。

<div align="center">
 <img src="./images/t5-finetune-summarization.svg" alt="图10.9.4 T5 微调用于摘要（英文版 Fig. 11.9.4）" width="400">
 <br><small>图10.9.4 T5 微调用于摘要（英文版 Fig. 11.9.4）</small>
</div>

以文本摘要为例说明了T5的微调。在这个下游任务中，任务描述词元"Summarize"、":"后跟文章词元被输入到编码器。

微调之后，拥有110亿参数的T5（T5-11B）在多个编码（例如分类）和生成（例如摘要）基准上取得了最先进的结果。自发布以来，T5已被后续研究广泛使用。例如，switch Transformer基于T5设计，通过激活参数的子集来获得更好的计算效率 ([Fedus et al., 2022](https://zh.d2l.ai/chapter_references/zreferences.html#fedus2022switch))。在一个名为Imagen的文本到图像模型中，文本被输入到拥有46亿参数的冻结T5编码器（T5-XXL） ([Saharia et al., 2022](https://zh.d2l.ai/chapter_references/zreferences.html#saharia2022photorealistic))。

<div align="center">
 <img src="./images/imagen.png" alt="图10.9.5 Imagen（英文版 Fig. 11.9.5）" width="400">
 <br><small>图10.9.5 Imagen（英文版 Fig. 11.9.5）</small>
</div>

图10.9.5中的逼真文本到图像示例表明，即使没有微调，仅T5编码器也可以有效地表示文本。

## 10.9.3.仅解码器

我们已经回顾了仅编码器和编码器-解码器Transformer。或者，仅解码器Transformer从图10.7.1中描绘的原始编码器-解码器架构中移除整个编码器以及包含编码器-解码器交叉注意力的解码器子层。

<div align="center">
 <img src="./images/transformer.svg" alt="图10.7.1 Transformer 架构" width="400">
 <br><small>图10.7.1 Transformer 架构</small>
</div>

如今，仅解码器Transformer已成为大规模语言建模（见[8.3节](../08_pypto_recurrent_neural_networks/08.03_language_models_and_dataset.ipynb)）中*事实上的*（de facto）架构，它通过自监督学习利用世界上丰富的未标注文本语料库。



### 10.9.3.1.GPT和GPT-2

GPT（生成式预训练，generative pre-training）模型使用语言建模作为训练目标，选择Transformer解码器作为其骨干网络 ([Radford et al., 2018](https://zh.d2l.ai/chapter_references/zreferences.html#Radford.Narasimhan.Salimans.ea.2018))。

按照[8.3节](../08_pypto_recurrent_neural_networks/08.03_language_models_and_dataset.ipynb)中的分区序列描述的自回归语言模型训练方式，图10.9.6展示了GPT使用Transformer解码器的预训练，其中目标序列是输入序列向右移动一个词元。

<div align="center">
 <img src="./images/gpt-decoder-only.svg" alt="图10.9.6 GPT 语言建模预训练（英文版 Fig. 11.9.6）" width="400">
 <br><small>图10.9.6 GPT 语言建模预训练（英文版 Fig. 11.9.6）</small>
</div>

请注意，Transformer解码器中的注意力模式强制每个词元只能关注其过去的词元（未来词元无法被关注，因为它们尚未被选择）。

GPT拥有1亿参数，需要针对各个下游任务进行微调。一年后，一个更大的Transformer解码器语言模型GPT-2被提出 ([Radford et al., 2019](https://zh.d2l.ai/chapter_references/zreferences.html#Radford.Wu.Child.ea.2019))。与GPT中原始的Transformer解码器相比，GPT-2采用了预归一化（在[10.8节](10.08_vision_transformer.ipynb)中的视觉Transformer编码器中讨论）以及改进的初始化和权重缩放。GPT-2在40 GB文本上预训练，拥有15亿参数，在语言建模基准上取得了最先进的结果，并且在多个其他任务上取得了有前景的结果，*而无需更新参数或架构*。



### 10.9.3.2.GPT-3及以后

GPT-2展示了在不更新模型的情况下将同一个语言模型用于多个任务的潜力。这比微调在计算上更高效，因为微调需要通过梯度计算来更新模型。

在解释这种不更新参数、计算上更高效的语言模型使用方式之前，回顾[8.5节](../08_pypto_recurrent_neural_networks/08.05_rnn_scratch.ipynb)可知，语言模型可以在给定某个前缀文本序列的条件下被训练来生成文本序列。因此，预训练语言模型可以在*不更新参数*的情况下，以包含任务描述、特定任务的输入-输出示例以及提示（任务输入）的输入序列为条件，生成任务输出序列。这种学习范式被称为*上下文学习*（in-context learning） ([Brown et al., 2020](https://zh.d2l.ai/chapter_references/zreferences.html#brown2020language))，当有零个、一个或少量任务特定的输入-输出示例时，可以进一步分为*零样本*、*单样本*和*少样本*学习（图10.9.7）。

<div align="center">
 <img src="./images/gpt-3-xshot.svg" alt="图10.9.7 GPT-3 少样本学习（英文版 Fig. 11.9.7）" width="400">
 <br><small>图10.9.7 GPT-3 少样本学习（英文版 Fig. 11.9.7）</small>
</div>

这三种设置在GPT-3中得到了测试 ([Brown et al., 2020](https://zh.d2l.ai/chapter_references/zreferences.html#brown2020language))，其最大版本使用的数据和模型规模大约比GPT-2大两个数量级。GPT-3使用与其直接前身GPT-2相同的Transformer解码器架构，唯一不同的是注意力模式（图10.9.6的右侧）在交替层中更稀疏。GPT-3在3000亿词元上预训练，模型规模越大性能越好，其中少样本性能增长最快（图10.9.8）。

<div align="center">
 <img src="./images/gpt3-xshot-scaling.png" alt="图10.9.8 GPT-3 少样本性能随模型规模扩展（英文版 Fig. 11.9.8）" width="400">
 <br><small>图10.9.8 GPT-3 少样本性能随模型规模扩展（英文版 Fig. 11.9.8）</small>
</div>

随后的GPT-4模型在其报告中并未完全公开技术细节 ([OpenAI, 2023](https://zh.d2l.ai/chapter_references/zreferences.html#openai2023gpt4))。与其前身相比，GPT-4是一个大规模多模态模型，可以同时接收文本和图像作为输入，并生成文本输出。



## 10.9.4.可扩展性

图10.9.8实证证明了Transformer在GPT-3语言模型中的可扩展性。对于语言建模，对Transformer可扩展性更全面的实证研究让研究人员看到了用更多数据和算力训练更大Transformer的前景 ([Kaplan et al., 2020](https://zh.d2l.ai/chapter_references/zreferences.html#kaplan2020scaling))。

<div align="center">
 <img src="./images/scaling-power-law.png" alt="图10.9.9 缩放定律：损失随算力/数据/参数幂律下降（英文版 Fig. 11.9.9）" width="400">
 <br><small>图10.9.9 缩放定律：损失随算力/数据/参数幂律下降（英文版 Fig. 11.9.9）</small>
</div>

如图10.9.9所示，在模型规模（参数数量，不包括嵌入层）、数据集大小（训练词元数量）和训练算力量（PetaFLOP/s-天，不包括嵌入层）方面可以观察到*幂律扩展*（power-law scaling）。总的来说，同时增加这三个因素会带来更好的性能。然而，*如何*同时增加它们仍然是一个有争议的问题 ([Hoffmann et al., 2022](https://zh.d2l.ai/chapter_references/zreferences.html#hoffmann2022training))。 除了性能提升外，大模型比小模型享有更好的样本效率。

<div align="center">
 <img src="./images/scaling-sample-conv.png" alt="图10.9.10 样本效率随规模提升（英文版 Fig. 11.9.10）" width="400">
 <br><small>图10.9.10 样本效率随规模提升（英文版 Fig. 11.9.10）</small>
</div>

图10.9.10表明，大模型只需要更少的训练样本（处理的词元）就能达到与小模型相同的性能水平，并且性能随算力平滑扩展。([Kaplan et al., 2020](https://zh.d2l.ai/chapter_references/zreferences.html#kaplan2020scaling))中的经验缩放行为已在后续的大型Transformer模型中得到验证。例如，GPT-3在计算规模多两个数量级时的验证支持了这一假设（图10.9.11）。

<div align="center">
 <img src="./images/scaling-gpt3.png" alt="图10.9.11 GPT-3 规模扩展（英文版 Fig. 11.9.11）" width="400">
 <br><small>图10.9.11 GPT-3 规模扩展（英文版 Fig. 11.9.11）</small>
</div>



## 10.9.5.大语言模型

GPT系列中Transformer的可扩展性启发了后续的大语言模型。
GPT-2的Transformer解码器被用于训练拥有5300亿参数的Megatron-Turing NLG ([Smith et al., 2022](https://zh.d2l.ai/chapter_references/zreferences.html#smith2022using))，使用了2700亿训练词元。遵循GPT-2的设计，拥有2800亿参数的Gopher ([Rae et al., 2021](https://zh.d2l.ai/chapter_references/zreferences.html#rae2021scaling))在3000亿词元上预训练，在各种任务上表现出竞争力。
Chinchilla ([Hoffmann et al., 2022](https://zh.d2l.ai/chapter_references/zreferences.html#hoffmann2022training))继承了相同的架构并使用与Gopher相同的算力预算，是一个规模小得多（700亿参数）但训练时间长得多（1.4万亿训练词元）的模型，在许多任务上胜过Gopher，并且更强调词元数量而非参数数量。
为了继续语言建模的扩展路线，
PaLM（路径语言模型，Pathway Language Model） ([Chowdhery et al., 2022](https://zh.d2l.ai/chapter_references/zreferences.html#chowdhery2022palm))是一个拥有5400亿参数的Transformer解码器，采用修改后的设计，在7800亿词元上预训练，在BIG-Bench基准上超过了人类的平均水平 ([Srivastava et al., 2022](https://zh.d2l.ai/chapter_references/zreferences.html#srivastava2022beyond))。其后续版本PaLM 2 ([Anil et al., 2023](https://zh.d2l.ai/chapter_references/zreferences.html#anil2023palm))以大约1:1的比例扩展数据和模型，并改进了多语言和推理能力。
其他大语言模型，例如进一步训练通才模型（PaLM）的Minerva ([Lewkowycz et al., 2022](https://zh.d2l.ai/chapter_references/zreferences.html#lewkowycz2022solving))和不在通用语料库上训练的Galactica ([Taylor et al., 2022](https://zh.d2l.ai/chapter_references/zreferences.html#taylor2022galactica))，已经展现出有前景的定量和科学推理能力。

开源发布，例如OPT（开放预训练Transformer，Open Pretrained Transformers） ([Zhang et al., 2022](https://zh.d2l.ai/chapter_references/zreferences.html#zhang2022opt))、BLOOM ([Scao et al., 2022](https://zh.d2l.ai/chapter_references/zreferences.html#scao2022bloom))和FALCON ([Penedo et al., 2023](https://zh.d2l.ai/chapter_references/zreferences.html#penedo2023refinedweb))，使大语言模型的研究和使用得以普及。
关注推理时的计算效率，开源的Llama 1 ([Touvron et al., 2023](https://zh.d2l.ai/chapter_references/zreferences.html#touvron2023llama))通过在比通常使用的更多词元上训练，超越了规模大得多的模型。更新后的Llama 2 ([Touvron et al., 2023](https://zh.d2l.ai/chapter_references/zreferences.html#touvron2023llama2))进一步将预训练语料库增加了40%，从而产生了可以与有竞争力的闭源模型性能相匹配的产品级模型。

([Wei et al., 2022](https://zh.d2l.ai/chapter_references/zreferences.html#wei2022emergent))讨论了大语言模型的涌现能力，这些能力在较大的模型中存在，但在较小的模型中不存在。
然而，简单地增加模型规模并不会让模型天然更好地遵循人类指令。
([Wei et al., 2021](https://zh.d2l.ai/chapter_references/zreferences.html#wei2021finetuned), [Sanh et al., 2021](https://zh.d2l.ai/chapter_references/zreferences.html#sanh2021multitask))发现，在通过*指令*描述的一系列数据集上微调大语言模型，可以提高保留任务上的零样本性能。
使用*基于人类反馈的强化学习*（reinforcement learning from human feedback）， ([Ouyang et al., 2022](https://zh.d2l.ai/chapter_references/zreferences.html#ouyang2022training))微调了GPT-3以遵循各种指令。
继通过微调使语言模型与人类意图对齐的InstructGPT ([Ouyang et al., 2022](https://zh.d2l.ai/chapter_references/zreferences.html#ouyang2022training))之后，
[ChatGPT](https://chat.openai.com/)可以根据与人类的对话生成类似人类的回复（例如代码调试和创意写作），并且可以零样本执行许多自然语言处理任务 ([Qin et al., 2023](https://zh.d2l.ai/chapter_references/zreferences.html#qin2023chatgpt))。
([Bai et al., 2022](https://zh.d2l.ai/chapter_references/zreferences.html#bai2022constitutional))用模型输出取代了人类输入（例如人工标注数据），以部分自动化指令微调过程，这也被称为*基于AI反馈的强化学习*（reinforcement learning from AI feedback）。

大语言模型提供了一个激动人心的前景：通过构造文本输入，借助上下文学习诱导模型执行期望的任务，这也被称为*提示*（prompting）。
值得注意的是，
*思维链提示*（chain-of-thought prompting） ([Wei et al., 2022](https://zh.d2l.ai/chapter_references/zreferences.html#wei2022chain))是一种上下文学习方法，使用少样本的"问题、中间推理步骤、答案"演示，激发大语言模型解决数学、常识和符号推理任务的复杂推理能力。
对多条推理路径进行采样 ([Wang et al., 2023](https://zh.d2l.ai/chapter_references/zreferences.html#wang2023self))、多样化少样本演示 ([Zhang et al., 2023](https://zh.d2l.ai/chapter_references/zreferences.html#zhang2023automatic))以及将复杂问题分解为子问题 ([Zhou et al., 2023](https://zh.d2l.ai/chapter_references/zreferences.html#zhou2023least))都可以提高推理准确率。事实上，只需在每个答案之前加上"Let's think step by step"这样简单的提示，大语言模型甚至可以以不错的准确率执行*零样本*思维链推理 ([Kojima et al., 2022](https://zh.d2l.ai/chapter_references/zreferences.html#kojima2022large))。
即使是包含文本和图像的多模态输入，语言模型执行多模态思维链推理的准确率也高于仅使用文本输入 ([Zhang et al., 2023](https://zh.d2l.ai/chapter_references/zreferences.html#zhang2023multicot))。



## 10.9.6.小结

Transformer已被预训练为仅编码器（例如BERT）、编码器-解码器（例如T5）和仅解码器（例如GPT系列）三种形式。预训练模型可以通过模型更新（例如微调）或不更新（例如少样本）来适配执行不同的任务。Transformer的可扩展性表明，更大的模型、更多的训练数据和更多的训练算力有利于获得更好的性能。由于Transformer最初是为文本数据设计和预训练的，本节略微偏向自然语言处理。尽管如此，上述讨论的模型经常可以在跨多种模态的更近期模型中找到。例如，
（i）Chinchilla ([Hoffmann et al., 2022](https://zh.d2l.ai/chapter_references/zreferences.html#hoffmann2022training))被进一步扩展为Flamingo ([Alayrac et al., 2022](https://zh.d2l.ai/chapter_references/zreferences.html#alayrac2022flamingo))，一个用于少样本学习的视觉语言模型；
（ii）GPT-2 ([Radford et al., 2019](https://zh.d2l.ai/chapter_references/zreferences.html#Radford.Wu.Child.ea.2019))和视觉Transformer在CLIP（对比语言-图像预训练，Contrastive Language-Image Pre-training） ([Radford et al., 2021](https://zh.d2l.ai/chapter_references/zreferences.html#radford2021learning))中编码文本和图像，其图像和文本嵌入后来被用于DALL-E 2文本到图像系统 ([Ramesh et al., 2022](https://zh.d2l.ai/chapter_references/zreferences.html#ramesh2022hierarchical))。虽然目前还没有关于多模态预训练中Transformer可扩展性的系统性研究，但一个名为Parti的全Transformer文本到图像模型 ([Yu et al., 2022](https://zh.d2l.ai/chapter_references/zreferences.html#yu2022scaling))展示了跨模态扩展的潜力：
更大的Parti更擅长高保真图像生成和内容丰富的文本理解（图10.9.12）。

<div align="center">
 <img src="./images/parti.png" alt="图10.9.12 Parti（英文版 Fig. 11.9.12）" width="400">
 <br><small>图10.9.12 Parti（英文版 Fig. 11.9.12）</small>
</div>


## 10.9.7.练习

1. 是否可以使用由不同任务组成的小批量来微调T5？为什么可以或不可以？GPT-2呢？
1. 给定一个强大的语言模型，你能想到哪些应用？
1. 假设要求你通过添加额外层来微调一个语言模型以执行文本分类。你会把层加在哪里？为什么？
1. 考虑序列到序列问题（例如机器翻译），其中输入序列在整个目标序列预测期间始终可用。使用仅解码器Transformer建模可能有什么局限性？为什么？

参考答案详见 [answers/10.09_reference_answer](./answers/10.09_reference_answer.ipynb)。


### 10.9.7.1.参考答案


In [ ]:
!cat answers/txt/10.09_reference_answer.txt